# Ideas planner, second list, idea 6: refine scale and heading by correlation at the candidate
When the height metadata is 20 % too high the standard scales (0.95-1.05) miss, and the wider scales (0.80-1.05) recover about 70 %. Here, at the best candidate of the pooled wide-scale queries, search scale (0.75-1.30 step 0.05) and heading (-10..+10 step 2.5) for the pose where the edge correlation at that place is highest (the whole canvas is used, not 5 inliers), re-render once at that pose and run LoFTR once.
Photos: the 60 official photos of Test_Iteration_9 (dumps official60_scale15_clean and official60_scale15_height20, 15 queries: headings -5/0/+5 x scales 0.80..1.05). Reading: points of the current rule on the dump, and with the refined pose; clean answers must not change.
Falsification (planned): dead if the refinement recovers less than 5 of the 11 points still missing under height +20 %, or changes any clean answer to a worse one.

In [1]:
import os, sys, time, gc, functools
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
T7 = ROOT / "tests/Test_Iteration_7/results"

DATA = os.environ.get("GATE_DATA", "official")          # official or nudt
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))
TAG = os.environ.get("TAG", "official150")              # which rich dump (results/rich_dump/<TAG>) names the photos
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))           # >0: only the first photos (smoke test)
print(DATA, TAG, "pool", POOL_PHOTOS, "limit", N_LIMIT)

official official150 pool 500 limit 0


In [2]:
import cv2
os.environ.setdefault("PHOTOS_FILE", str(ROOT / "tests/Test_Iteration_9/results/photos_official.csv"))

'/home/voidness/Storage/Synced Folders/InterIIT Tech 15/IdeaForge-GeoLoc-InterIIT15/tests/Test_Iteration_9/results/photos_official.csv'

In [3]:
if DATA == "nudt":
    from train.datasets import photo_sets
    photo_sets.load_nudt_photos = functools.partial(photo_sets.load_nudt_photos, split=os.environ.get("NUDT_SPLIT", "test"))
from train.checks.rich_dump import RichDumpPipeline
from train.checks import replay
from coarse_search.heading_queries import query_geometry
from image_processing.geometry import render_query
from image_processing.map_levels import level_for_height, level_gsd

def make_pipeline(headings, scales, confidence=0.22, coarse_threshold=0.21):
    pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                                matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=confidence, satellite="new",
                                map_margin_m=None, loftr_coarse_threshold=coarse_threshold, dino_weight=None, headings=headings, scale_factors=scales)
    start = time.time()
    pipeline.setup(POOL_PHOTOS, 7)
    print("setup %.0f s" % (time.time() - start))
    return pipeline

STANDARD = [(-5.0, 1.0), (0.0, 1.0), (5.0, 1.0), (-5.0, 1.05), (0.0, 1.05), (5.0, 1.05)]
dump = None

In [4]:
pipeline = make_pipeline([-5.0, 0.0, 5.0], [0.80, 0.85, 0.90, 1.0, 1.05])
from train.checks import part_maps
from train.checks.perturb import perturbed_item
from image_processing.thermal import load_thermal_frame
from fine_matching.loftr_matcher import candidate_matches, stage2_from_matches
test_set = pipeline.test_set
indices = sorted(int(i) for i in pd.read_csv(os.environ["PHOTOS_FILE"]).photo)
if N_LIMIT:
    indices = indices[:N_LIMIT]
SCALES = [round(0.75 + 0.05 * i, 2) for i in range(12)]
HEADINGS = [-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0]
sets = {kind: replay.load_set(T7 / "rich_dump" / ("official60_scale15_" + kind)) for kind in ("clean", "height20")}
KEYS = [(h, s) for s in (0.80, 0.85, 0.90, 1.0, 1.05) for h in (-5.0, 0.0, 5.0)]
print({k: len(v) for k, v in sets.items()}, len(indices), "photos")

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 111 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup 126 s


{'clean': 60, 'height20': 60} 60 photos


In [5]:
rows = []
start = time.time()
for kind in ["clean", "height20"]:
    for index in indices:
        if index not in sets[kind]:
            continue
        photo = sets[kind][index]
        item = dict(test_set.items[index]) if kind == "clean" else perturbed_item(test_set.items[index], "height20", "", load_thermal_frame)
        level = photo["level"]; truth = photo["truth"]; sigma = photo["sigma"]
        ids = replay.query_ids(photo, KEYS)
        cand = replay.pooled(photo, ids)
        k = replay.choose(cand, 1.0)
        base = dict(kind=kind, photo=index, height=photo["height"], inliers_dump=float(cand["inliers"][k]) if k >= 0 else 0.0)
        if k >= 0:
            x0, y0 = float(cand["x"][k]), float(cand["y"][k])
            error_dump = replay.error_sigma(photo, x0, y0)
            base.update(err_dump=error_dump, points_dump=replay.points(error_dump) if cand["inliers"][k] >= 8 else 0)
        else:
            base.update(err_dump=np.nan, points_dump=0)
        if k < 0 or cand["inliers"][k] < 1:
            rows.append(dict(base, refined_inliers=0.0, err_refined=np.nan, points_refined=0, best_scale=np.nan, best_heading=np.nan)); continue
        best = (-2.0, None, None)
        for s in SCALES:
            for h in HEADINGS:
                canvas, valid = render_query(item, pipeline.camera, level, query_geometry(h, s))
                value, _, _ = part_maps.local_peak(pipeline.store, canvas, valid, level, x0, y0, 3, None)
                if value > best[0]:
                    best = (value, h, s)
        canvas, valid = render_query(item, pipeline.camera, level, query_geometry(best[1], best[2]))
        peaks = np.array([[1.0, x0, y0, 0.0, 0.0]])
        stored = candidate_matches(pipeline.matcher, canvas, valid, level, peaks, pipeline.level_images)
        second = stage2_from_matches(stored, level, 0.22)
        x1, y1 = x0 + second["shifts"][0][0], y0 + second["shifts"][0][1]
        error_refined = replay.error_sigma(photo, x1, y1)
        inl = float(second["inliers"][0])
        rows.append(dict(base, refined_inliers=inl, err_refined=error_refined, points_refined=replay.points(error_refined) if inl >= 8 else 0,
                         best_scale=best[2], best_heading=best[1], best_value=best[0]))
        torch.cuda.empty_cache()
    print(kind, "done %.0f s" % (time.time() - start), flush=True)
R = pd.DataFrame(rows)
R["points_best"] = np.maximum(R.points_dump, R.points_refined)
R.to_csv(T7 / "idea6b_refine.csv", index=False)
summary = R.groupby("kind").agg(photos=("photo", "count"), score_dump=("points_dump", lambda p: round(p.sum() / len(p) * 6, 2)),
                                score_refined_only=("points_refined", lambda p: round(p.sum() / len(p) * 6, 2)),
                                score_best_of_both=("points_best", lambda p: round(p.sum() / len(p) * 6, 2))).reset_index()
summary.to_csv(T7 / "idea6b_refine_summary.csv", index=False)
print(summary.to_string(index=False))

clean done 66 s


height20 done 130 s


    kind  photos  score_dump  score_refined_only  score_best_of_both
   clean      60        47.8                47.8                50.6
height20      60        44.5                41.6                46.0
